# Project Start

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sbn
from altair import Chart, X, Y, Color, Scale
import altair as alt
from vega_datasets import data
import requests
from bs4 import BeautifulSoup

In [ ]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [ ]:
wd = pd.read_csv('shot_logs.csv')

FileNotFoundError: ignored

# Background: The NBA collects a lot of data on how players play and I want to find out how we can use this data to optimize play.

## Questions: 
Is there a correlation between how far away from the hoop a player is and the result of the shot? <br>
Is there a correlation between how many dribbles and the result of the shot? <br>
Is there a correlation between how far away the nearest defender is and the result of the shot? <br>
Can we use this data to find the top 3 shooters in the league and how to counter them? <br>
Is it worth it to take a 2 point shot or a 3 point shot? <br>

In [ ]:
wd.head()

In [ ]:
wd['SHOT_DIST'] = pd.to_numeric(wd['SHOT_DIST'], errors='coerce')

In [ ]:
wd['SHOT_DIST']

## Is there a correlation between the distance of the shot and shot result

### All shots by distance

In [ ]:
wd.SHOT_DIST

In [ ]:
wd.SHOT_DIST.mean()

In [ ]:
Chart(wd).mark_bar().encode(x=X('SHOT_DIST', bin=True), y='count()')

In [ ]:
ms = wd[wd.SHOT_RESULT == 'made']

NameError: ignored

### All shots by distance that scored points

In [ ]:
Chart(ms).mark_bar().encode(x=X('SHOT_DIST', bin=True), y='count()')

In [ ]:
twoptsmade = ms[ms.PTS_TYPE == 2]
twoptsmade.shape

In [ ]:
twoptsshots = wd[wd.PTS_TYPE == 2]
twoptsshots.shape

### 45990 out of 94173 shots were made that got 2 points 


In [ ]:
45990 * 2

In [ ]:
threeptsmade = ms[ms.PTS_TYPE == 3]
threeptsmade.shape

In [ ]:
threeptsshots = wd[wd.PTS_TYPE == 3]
threeptsshots.shape

### 11915 out of 33896 shots were made that got 3 points

In [ ]:
11915 * 3

### because of how accesable it is to make and take a 2 point shot it doesn't really seem worth it to take a three point shot

In [ ]:
dms = wd[wd.SHOT_RESULT == 'missed']

### All shots by distance that didn't score any points

In [ ]:
Chart(dms).mark_bar().encode(x=X('SHOT_DIST', bin=True), y='count()').interactive()

In [ ]:
corr = ms.corr()
sbn.heatmap(corr, 
            xticklabels=corr.columns.values,
            yticklabels=corr.columns.values)

### DONT RUN NEXT CELL CRASHES NOTEBOOK

In [ ]:
#alt.Chart(source).mark_bar().encode(
#    x=X('SHOT_DIST', bin=True),
#    y='count()',
#    color='SHOT_RESULT'
#)

In [ ]:
wd.player_name

### Shots that Brian Roberts made, organized by distance

In [ ]:
brshots = wd[wd.player_name == 'brian roberts']

In [ ]:
brmadeshots = brshots[brshots.SHOT_RESULT == 'made']

In [ ]:
Chart(brmadeshots).mark_bar().encode(x=X('SHOT_DIST', bin=True), y='count()')

### Organized by amount of dribbles before shooting

In [ ]:
Chart(brmadeshots).mark_bar().encode(x=X('DRIBBLES', bin=True), y='count()')

### Organized by Closest Defender

In [ ]:
Chart(brmadeshots).mark_bar().encode(x=X('CLOSE_DEF_DIST', bin=True), y='count()')

### From this we can tell that the perfect shot for Brian Roberts would be, 20 to 25 feet from the hoop, the nearest defender 2 to 6 feet away, and with very little dribbles before the shot

In [ ]:
pd.set_option('display.max_row', 10000)

# Set iPython's max column width to 500
pd.set_option('display.max_columns', 500)

In [ ]:
list_of_players = []
newdict ={}
for i in wd.player_name.unique():
   list_of_players.append(i)
#print(list_of_players)
for i in list_of_players:
    temp = ms[ms.player_name == i]
    #print(i)
    #print(len(temp))
    newdict[i] = len(temp)
a = sorted(newdict.items(), key=lambda x: x[1],reverse=True)
print(a[:10])
top10players = ['nikola vucevic', 'lebron james', 'james harden', 'mnta ellis', 'lamarcus aldridge', 'stephen curry', 'anthony davis', 'klay thompson', 'blake griffin', 'kyrie irving']
print(top10players)

### From this we can count how many shots each player made and scored points. After this we organize them by who made the most. This gives us our top 10 scorers.

In [ ]:
ms[ms.player_name == 'lebron james'].shape

### How can we counter the top 3 players?

In [ ]:
nv = ms[ms.player_name == 'nikola vucevic']

chart1 = alt.Chart(nv).mark_bar().encode(
    x=X('SHOT_DIST', bin=True),
    y='count()',
).properties(
    height=300,
    width=300
)

chart2 = alt.Chart(nv).mark_bar().encode(
    x=X('DRIBBLES', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)


chart3 = alt.Chart(nv).mark_bar().encode(
    x=X('CLOSE_DEF_DIST', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)
chart1 | chart2 | chart3

## Nikola Vucevic likes to take shots close to the hoop, with little to no movement, and with a defender very close to him. If you were defending against Nikola, you would want to keep him far away from the hoop and make him move by playing very aggressive.

In [ ]:
nv = ms[ms.player_name == 'lebron james']

chart1 = alt.Chart(nv).mark_bar().encode(
    x=X('SHOT_DIST', bin=True),
    y='count()',
).properties(
    height=300,
    width=300
)

chart2 = alt.Chart(nv).mark_bar().encode(
    x=X('DRIBBLES', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)


chart3 = alt.Chart(nv).mark_bar().encode(
    x=X('CLOSE_DEF_DIST', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)
chart1 | chart2 | chart3

## Lebron James can make shots from anywhere on the court and likes to move around. He also likes to play with a defender on top of him. To counter a player like this, the defender should not let Lebron get close to the hoop and stop him from moving, basically lock him down.

In [ ]:
nv = ms[ms.player_name == 'james harden']

chart1 = alt.Chart(nv).mark_bar().encode(
    x=X('SHOT_DIST', bin=True),
    y='count()',
).properties(
    height=300,
    width=300
)

chart2 = alt.Chart(nv).mark_bar().encode(
    x=X('DRIBBLES', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)


chart3 = alt.Chart(nv).mark_bar().encode(
    x=X('CLOSE_DEF_DIST', bin=True),
    y='count()'
).properties(
    height=300,
    width=300
)
chart1 | chart2 | chart3

## James Harden is most definetly a shooter. He likes to move around and take shots from very far away, and he can also hit those shots. To counter a player like this, play on top of him and make him move up to the hoop.

### link to data: https://www.kaggle.com/dansbecker/nba-shot-logs